# Monarch - TRIBE v2 batch NAA scans (Kaggle GPU)

Runs the full text->fMRI cascade (TTS -> WhisperX -> Llama-3.2-3B embeddings -> TRIBE v2) and the batch NAA + alpha calibration for the research paper.

**Before running:** Settings -> Accelerator = GPU (T4/P100). Add-ons -> Secrets -> `HF_TOKEN` (must have accepted the Llama-3.2 licence on HF).

In [ ]:
import torch
print(torch.__version__, 'cuda:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU-only')

In [ ]:
%%bash
mkdir -p /content/monarch_run
cd /content/monarch_run
rm -rf monarch tribev2
git clone -q https://github.com/brn-mwai/monarch.git
git clone -q https://github.com/brn-mwai/tribev2.git
apt-get -qq update && apt-get -qq install -y ffmpeg > /dev/null
echo cloned

In [ ]:
%%bash
pip install -q exca==0.5.17
pip install -q nibabel ujson mne torchmetrics x_transformers gtts langdetect spacy python-Levenshtein
pip install -q whisperx --no-deps
pip install -q pyannote.audio "torch==2.11.0" "torchaudio" "numpy==2.0.2"
pip install -q faster-whisper ctranslate2 nltk
pip install -q neuralset==0.0.2 neuraltrain==0.0.2 --no-deps
pip install -q /content/monarch_run/tribev2 --no-deps
python -m spacy download en_core_web_lg -q
echo deps done

In [ ]:
import re, site, pathlib
sp = site.getsitepackages()[0]
ns = pathlib.Path(sp, "neuralset", "__init__.py")
if ns.exists():
    ns.write_text(ns.read_text().replace('_XK_MIN = "0.5.20"', '_XK_MIN = "0.5.17"'))
    print("neuralset guard lowered")

hp = pathlib.Path(sp, "exca", "helpers.py")
if hp.exists():
    src = hp.read_text()
    if "_get_discriminated_subclasses" not in src:
        src += """

def _get_subclasses(cls):
    out = []
    for c in cls.__subclasses__():
        out.append(c)
        out.extend(_get_subclasses(c))
    return out

def _apply_discriminator_patch():
    import exca.helpers as H
    for name in dir(H):
        obj = getattr(H, name)
        if isinstance(obj, type) and hasattr(obj, "_exca_discriminator_key"):
            obj._get_discriminated_subclasses = classmethod(
                lambda cls: {c.__name__: c for c in [cls] + H._get_subclasses(cls)})
            obj.__new__ = staticmethod(lambda cls, *a, **k: object.__new__(cls))

_apply_discriminator_patch()
"""
        hp.write_text(src)
        print("exca helpers patched")
    else:
        print("exca helpers already patched")
else:
    print("exca helpers.py not found (skipped)")


et = pathlib.Path("/content/monarch_run/tribev2/tribev2/eventstransforms.py")
if et.exists():
    s = et.read_text()
    if "--vad_method" not in s:
        s = s.replace('                "--output_dir",',
                      '                "--vad_method",\n                "silero",\n                "--output_dir",')
        et.write_text(s)
        print("eventstransforms: silero VAD injected")
    else:
        print("eventstransforms already patched")
else:
    print("eventstransforms.py not found (skipped)")


In [ ]:
from google.colab import userdata
import os
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
os.environ['HF_HUB_DISABLE_XET'] = '1'
os.environ['MONARCH_WHISPER_MODEL'] = 'small'
os.environ['MONARCH_WHISPER_DEVICE'] = 'cuda'
os.environ['MONARCH_WHISPER_COMPUTE'] = 'float16'
os.environ['MONARCH_WHISPERX_CMD'] = 'whisperx'
os.environ['PYTHONPATH'] = '/content/monarch_run/tribev2'
print('env set')

In [ ]:
%%bash
cd /content/monarch_run/monarch/services/inference
PYTHONPATH=/content/monarch_run/tribev2 python scripts/smoke_test.py

Expect: `Model loaded on device: cuda:0`, `Predictions shape: (T, 20484)`, `Smoke test PASSED`.

In [ ]:
import csv, io, urllib.request
base = 'https://raw.githubusercontent.com/JULIELab/EmoBank/master/corpus/'
def fetch(name):
    return list(csv.DictReader(io.StringIO(urllib.request.urlopen(base + name).read().decode('utf-8'))))
texts = {r['id']: r['text'] for r in fetch('emobank.csv')}
writer_rows = fetch('writer.csv')
vals = [float(r['A']) for r in writer_rows if r.get('A')]
lo, hi = min(vals), max(vals)
with open('/content/monarch_run/emobank_raw.csv', 'w', newline='', encoding='utf-8') as h:
    w = csv.DictWriter(h, fieldnames=['text', 'label2'])
    w.writeheader()
    n = 0
    for r in writer_rows:
        t = texts.get(r['id'], '').strip()
        if t and r.get('A'):
            w.writerow({'text': t, 'label2': f"{(float(r['A'])-lo)/(hi-lo):.4f}"})
            n += 1
print('rows:', n, 'writer arousal range:', lo, hi)

In [ ]:
%%bash
cd /content/monarch_run/monarch/services/inference
python scripts/build_emobank_corpus.py --raw /content/monarch_run/emobank_raw.csv --out /content/monarch_run/emobank_corpus.csv --per-bin 8
PYTHONPATH=/content/monarch_run/tribev2 python scripts/batch_naa.py --csv /content/monarch_run/emobank_corpus.csv --text-col text --outcome-col arousal --out /content/monarch_run/corpus_naa.csv
python scripts/calibrate_alpha.py --csv /content/monarch_run/corpus_naa.csv --naa-col naa --outcome-col arousal --out /content/monarch_run/alpha_hat.json || true
cat /content/monarch_run/alpha_hat.json

**Paper rule:** if the alpha CI straddles 0 (it did on MI300X: alpha_hat -0.1321, CI [-0.92, 0.66], holdout R2 -0.162) report the NULL RESULT - do not quote an alpha_hat. Signed NAA (A_aff - A_del) is the metric; classification bands on sign only.

Download everything in `/content/monarch_run` (right panel -> Output) when done.